# MEIA Challenge 3 — EDA

Análise exploratória dos datasets de logs Sysmon.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from preprocessor import parse_csv, make_windows
from detectors import IForestDetector, tag_techniques


## 1. Carregar dados

In [ ]:
df = parse_csv(Path('../data/samples/sample_lmd.csv'), dataset='lmd')
print(f'Events: {len(df)}')
print(f'Time range: {df.timestamp.min()} → {df.timestamp.max()}')
df.head()

## 2. Distribuição de EventIDs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['event_id'].value_counts().plot(kind='bar', ax=axes[0], title='EventID distribution')
df['process_name'].value_counts().head(15).plot(kind='barh', ax=axes[1], title='Top processes')

plt.tight_layout()
plt.show()

## 3. Windowing e features

In [ ]:
windows = list(make_windows(df, window_size_seconds=60))
print(f'Windows: {len(windows)}')

wdf = pd.DataFrame([w.to_dict() for w in windows])
wdf[['event_count', 'suspicious_process_count', 'network_connection_count',
     'powershell_count', 'has_mimikatz', 'has_psexec']].describe()

## 4. IsolationForest scoring

In [ ]:
import numpy as np

X = np.array([w.to_feature_vector() for w in windows])
iforest = IForestDetector()
iforest.fit(X)
scores = iforest.score(X)

wdf['if_score'] = scores

plt.figure(figsize=(12, 3))
plt.bar(range(len(scores)), scores, color=['red' if s > 0.6 else 'steelblue' for s in scores])
plt.axhline(0.6, color='orange', linestyle='--', label='Threshold 0.6')
plt.xlabel('Window index')
plt.ylabel('Anomaly score')
plt.title('IsolationForest anomaly scores per window')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Windows above threshold: {(scores > 0.6).sum()}/{len(scores)}')

## 5. ATT&CK rule tagging

In [ ]:
for i, w in enumerate(windows):
    hits = tag_techniques(w.to_dict())
    if hits:
        print(f'Window {i} (score={scores[i]:.2f}):')
        for h in hits:
            print(f'  {h["technique"]} {h["name"]} conf={h["confidence"]}')

## 6. Análise dos resultados do judge (se existirem)

In [ ]:
results_path = Path('../results')
judge_files = list(results_path.rglob('judge_results.json'))

if judge_files:
    with open(sorted(judge_files)[-1]) as f:
        judge_data = json.load(f)

    jdf = pd.DataFrame(judge_data)
    print(f'Judge results: {len(jdf)}')
    print(jdf[['window_start', 'anomaly_score', 'verdict', 'fp_risk']].to_string())
else:
    print('Nenhum ficheiro de judge results encontrado. Corre o pipeline primeiro.')